In [2]:
!!pip install litellm

# Important!!!
#
# <---- Set your 'GEMINI_API_KEY' as a secret over there with the "key" icon
#
#
import os
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')
os.environ['GEMINI_API_KEY'] = api_key

In [3]:

from google import genai
from typing import List,Dict

client=genai.Client()
def generate_response(prompt: List[Dict]) -> str:
    texts="\n".join(
        messages['content']
        for messages in prompt)
    res = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=texts,
    )

    return res.text



In [12]:
import json

def extract_markdown_block(response,par):
  if "```" not in response:
    return response.strip()
  block=response.split("```")[1].strip()
  return block[(len("action")):].strip()

def parse_action(response: str)->Dict:
  try:
    response=extract_markdown_block(response,"action")
    response_json=json.loads(response)
    if "tool_name" in response_json and "args" in response_json:
      return response_json
    else :
      return {"tool_name":"error","args":{"message":"you must respond with a json tool invocation."}}

  except json.JSONDecodeError:
    return {"tool_name":"error","args":{"message":"Invalid json response,you must respond with a json too invocatiion."}}



def list_files()-> List[str]:
  return os.listdir(".")

def read_file(file_name:str)->str:
  try :
     with open(file_name, "r") as file:
        return file.read()
  except FileNotFoundError:
    return f"Error: {file_name} not found"
  except Exception as e:
    return f"Error: {str(e)}"


agent_rules=[{
    "role":"system",
    "content":"""
    You are a AI agent that perform task with these avaliable tools.
    Avaliable tools:
     ```json
     "list_files":{
        "description":"returns all files in the current directory",
        "parameter":{}
     },
     "read_file":{
     "description":"description": "Reads the content of a file.",
        "parameters": {
            "file_name": {
                "type": "string",
                "description": "The name of the file to read."
            }
        }
     }
     "terminate": {
        "description": "Ends the agent loop and provides a summary of the task.",
        "parameters": {
            "message": {
                "type": "string",
                "description": "Summary message to return to the user."
            }
        }
    }
    ```
    If a user asks about files, documents, or content, first list the files before reading them.

When you are done, terminate the conversation by using the "terminate" tool and I will provide the results to the user.
    important!! every response must be an action,
    you must always responde this formate,
    <Stop and think step by step.parameter map to args,Insert a rich description of your step by step throught here>
    ```action
    {
    "tool_name":"Insert a tool name here",
    "args":{fill in any required arguments here}
    }
    ```

    """
}]


In [13]:
# Initialize agent parameters
iterations = 0
max_iterations = 3

user_task = input("What would you like me to do? ")

memory = [{"role": "user", "content": user_task}]
while iterations<max_iterations:
  # 1. Construct prompt: Combine agent rules with memory
    prompt = agent_rules + memory
  # 2. Generate response from LLM
    print("Agent Thinking")
    response = generate_response(prompt)
    print(f"Agent response: {response}")
  # 3. Parse response to determine action
    action = parse_action(response)
    result = "Action executed"

    if action["tool_name"]=="list_files":
      result={"result":list_files()}
    elif action["tool_name"] == "read_file":
      result={"result":read_file(action["args"]["file_name"])}
    elif action["tool_name"] == "error":
        result = {"error": action["args"]["message"]}
    elif action["tool_name"] == "terminate":
        print(action["args"]["message"])
        break
    else:
        result = {"error": "Unknown action: " + action["tool_name"]}
    print(f"Action result: {result}")
  # 5. Update memory with response and results
    memory.extend([
        {"role":"assistant","content":response},
        {"role":"user","content":json.dumps(result)}
    ])
 #6. Check termination condition
    if action["tool_name"] == "terminate":
        break
    iterations += 1

What would you like me to do? tell me what is in the directory
Agent Thinking
Agent response: <Stop and think step by step. The user wants to know what files are in the current directory. I need to list the files using the `list_files` tool first.>
```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action result: {'result': ['.config', 'sample_data']}
Agent Thinking
Agent response: <Stop and think step by step. The `list_files` tool has returned the contents of the current directory, which are `.config` and `sample_data`. Since the user asked to know what is in the directory, I have gathered the necessary information and can now terminate the task by summarizing the result.>
```action
{
    "tool_name": "terminate",
    "args": {
        "message": "The current directory contains the following items: '.config' and 'sample_data'."
    }
}
```
The current directory contains the following items: '.config' and 'sample_data'.
